# Prompt Improvement Reviewer (Latest Run)

This notebook:
1. Loads the latest run-scoped eval artifacts.
2. Identifies relevant failed scenarios.
3. Sends each failure + prompt context to your `.env`-configured Azure OpenAI model.
4. Displays concrete prompt improvement recommendations.

In [1]:
from __future__ import annotations

import json
import os
import re
from pathlib import Path
from typing import Any

import pandas as pd
from dotenv import load_dotenv
from openai import AzureOpenAI
from IPython.display import display

load_dotenv()

_candidate_roots = [Path("eval/results"), Path("results")]
RESULTS_ROOT = next((p for p in _candidate_roots if p.exists()), Path("eval/results"))
TRACES_GLOB = "traces-*__model-*__temp-*__top-p-*"

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)

print(f"Using RESULTS_ROOT={RESULTS_ROOT.resolve() if RESULTS_ROOT.exists() else RESULTS_ROOT}")

Using RESULTS_ROOT=C:\Users\blaineperry\git\agent-framework-minimal-sql\eval\results


In [2]:
def _find_latest_run_dir(results_root: Path) -> Path:
    dirs = [p for p in results_root.glob(TRACES_GLOB) if p.is_dir()]
    if not dirs:
        raise FileNotFoundError(f"No run-scoped folders found under {results_root}")

    ts_pattern = re.compile(r"^traces-(\d{8}-\d{6})")

    def _key(p: Path):
        m = ts_pattern.match(p.name)
        ts = m.group(1) if m else ""
        return (ts, p.stat().st_mtime)

    return max(dirs, key=_key)


def _read_json(path: Path) -> dict[str, Any]:
    if not path.exists():
        return {}
    with path.open("r", encoding="utf-8") as f:
        obj = json.load(f)
    return obj if isinstance(obj, dict) else {}


def _read_jsonl(path: Path) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    if not path.exists():
        return rows
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            if isinstance(obj, dict):
                rows.append(obj)
    return rows


latest_run_dir = _find_latest_run_dir(RESULTS_ROOT)
print(f"Latest run folder: {latest_run_dir}")

foundry_eval_path = latest_run_dir / "foundry_eval_latest.json"
eval_run_meta_path = latest_run_dir / "eval_run_metadata.json"
trace_run_meta_path = latest_run_dir / "run_metadata.json"
trace_jsonl_path = latest_run_dir / "agent_runs.jsonl"

foundry_eval = _read_json(foundry_eval_path)
eval_run_meta = _read_json(eval_run_meta_path)
trace_run_meta = _read_json(trace_run_meta_path)
trace_rows = _read_jsonl(trace_jsonl_path)

print("Loaded artifacts:")
print(f" - {foundry_eval_path} (exists={foundry_eval_path.exists()})")
print(f" - {eval_run_meta_path} (exists={eval_run_meta_path.exists()})")
print(f" - {trace_run_meta_path} (exists={trace_run_meta_path.exists()})")
print(f" - {trace_jsonl_path} (exists={trace_jsonl_path.exists()}, rows={len(trace_rows)})")

Latest run folder: results\traces-20260305-175355__model-gpt-4.1-mini__temp-default__top-p-default
Loaded artifacts:
 - results\traces-20260305-175355__model-gpt-4.1-mini__temp-default__top-p-default\foundry_eval_latest.json (exists=True)
 - results\traces-20260305-175355__model-gpt-4.1-mini__temp-default__top-p-default\eval_run_metadata.json (exists=True)
 - results\traces-20260305-175355__model-gpt-4.1-mini__temp-default__top-p-default\run_metadata.json (exists=True)
 - results\traces-20260305-175355__model-gpt-4.1-mini__temp-default__top-p-default\agent_runs.jsonl (exists=True, rows=10)


In [3]:
eval_rows = foundry_eval.get("rows") or []
if not isinstance(eval_rows, list):
    eval_rows = []

trace_lookup_by_case = {str(r.get("case_id")): r for r in trace_rows if r.get("case_id")}

failure_candidates: list[dict[str, Any]] = []
for row in eval_rows:
    if not isinstance(row, dict):
        continue

    det = row.get("deterministic_metrics") or {}
    ai = row.get("ai_judge_metrics") or {}

    req_fail = det.get("required_tools_pass") is False
    seq_fail = det.get("expected_sequence_pass") is False
    forb_fail = det.get("forbidden_tools_pass") is False

    ai_fail = False
    for k in ["intent_resolution", "task_adherence", "tool_call_accuracy"]:
        section = ai.get(k) or {}
        result_field = section.get(f"{k}_result")
        if isinstance(result_field, str) and result_field.lower() == "fail":
            ai_fail = True

    if req_fail or seq_fail or forb_fail or ai_fail:
        case_id = str(row.get("case_id") or "")
        trace_row = trace_lookup_by_case.get(case_id, {})
        failure_candidates.append(
            {
                "case_id": case_id,
                "query": row.get("query"),
                "response": row.get("response"),
                "deterministic_metrics": det,
                "ai_judge_metrics": ai,
                "actual_tool_calls": row.get("actual_tool_calls") or [],
                "expected_tools": row.get("expected_tools") or [],
                "required_tools": row.get("required_tools") or [],
                "forbidden_tools": row.get("forbidden_tools") or [],
                "chat_profile": trace_row.get("chat_profile"),
                "prompt_logical_profile": trace_row.get("prompt_logical_profile"),
                "prompt_manifest": trace_row.get("prompt_manifest") or {},
                "tool_names_available": trace_row.get("tool_names_available") or [],
            }
        )

failures_df = pd.DataFrame([
    {
        "case_id": f.get("case_id"),
        "query": f.get("query"),
        "chat_profile": f.get("chat_profile"),
        "prompt_logical_profile": f.get("prompt_logical_profile"),
        "required_tools_pass": (f.get("deterministic_metrics") or {}).get("required_tools_pass"),
        "expected_sequence_pass": (f.get("deterministic_metrics") or {}).get("expected_sequence_pass"),
        "forbidden_tools_pass": (f.get("deterministic_metrics") or {}).get("forbidden_tools_pass"),
        "tool_f1": (f.get("deterministic_metrics") or {}).get("tool_f1"),
        "required_missing": ", ".join((f.get("deterministic_metrics") or {}).get("required_missing", [])),
        "forbidden_hit": ", ".join((f.get("deterministic_metrics") or {}).get("forbidden_hit", [])),
    }
    for f in failure_candidates
])

print(f"Found {len(failure_candidates)} relevant failed scenario(s) in latest run.")
display(failures_df)

Found 6 relevant failed scenario(s) in latest run.


,case_id,query,chat_profile,prompt_logical_profile,required_tools_pass,expected_sequence_pass,forbidden_tools_pass,tool_f1,required_missing,forbidden_hit
0,SQL-001,What is the operational readiness of UIC W45AA...,Tactical Readiness AI,sql,True,False,True,0.8000,,
1,SQL-004,Show me the columns in ai.unit_readiness_rollup.,Tactical Readiness AI,sql,True,True,True,0.6667,,
2,SQL-006,What is the readiness status for UIC W45AAA an...,Combat LogiGuard AI,hybrid,True,False,True,1.0000,,
3,SQL-007,How many units are in C3 right now?,Tactical Readiness AI,sql,True,False,True,0.8000,,
4,SQL-008,Tell me a joke about tanks.,Tactical Readiness AI,sql,True,True,True,1.0000,,
5,SQL-010,Is equipment readiness under 80% for UIC W45AA...,Tactical Readiness AI,sql,True,False,True,0.8000,,


In [4]:
_prompt_roots = [Path("config/prompts"), Path("../config/prompts")]
PROMPTS_ROOT = next((p for p in _prompt_roots if p.exists()), Path("config/prompts"))

prompt_files = {
    "system": PROMPTS_ROOT / "system.yaml",
    "tools": PROMPTS_ROOT / "tools.yaml",
    "eval": PROMPTS_ROOT / "eval.yaml",
}

prompt_file_text: dict[str, str] = {}
for key, path in prompt_files.items():
    if path.exists():
        prompt_file_text[key] = path.read_text(encoding="utf-8")
    else:
        prompt_file_text[key] = ""

prompt_versions = (
    eval_run_meta.get("prompt_versions")
    or trace_run_meta.get("prompt_versions")
    or {}
)

llm_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT", "")
llm_api_key = os.getenv("AZURE_OPENAI_API_KEY", "")
llm_api_version = os.getenv("AZURE_OPENAI_API_VERSION", "")
llm_deployment = os.getenv("AZURE_OPENAI_EVAL_DEPLOYMENT") or os.getenv("AZURE_OPENAI_MODEL", "")

print("Prompt files:")
for key, path in prompt_files.items():
    print(f" - {key}: {path} (exists={path.exists()}, chars={len(prompt_file_text.get(key, ''))})")

print("\nPrompt versions from latest run:")
print(json.dumps(prompt_versions, indent=2))
print("\nLLM config check:")
print(f" - endpoint set: {bool(llm_endpoint)}")
print(f" - api key set: {bool(llm_api_key)}")
print(f" - api version set: {bool(llm_api_version)}")
print(f" - deployment: {llm_deployment or '(missing)'}")

Prompt files:
 - system: ..\config\prompts\system.yaml (exists=True, chars=2456)
 - tools: ..\config\prompts\tools.yaml (exists=True, chars=5187)
 - eval: ..\config\prompts\eval.yaml (exists=True, chars=1017)

Prompt versions from latest run:
{
  "system": "tactical_readiness_system@1.0.0",
  "tools": "1.0.0",
  "eval": "1.0.0"
}

LLM config check:
 - endpoint set: True
 - api key set: True
 - api version set: True
 - deployment: gpt-4.1-mini


In [5]:
MAX_FAILURES_TO_ANALYZE = 5
RUN_LLM_RECOMMENDER = True

SYSTEM_INSTRUCTION = """
You are a senior prompt engineer for an Azure SQL + tool-using assistant.
Given failed evaluation scenarios, recommend prompt improvements.
Focus on behavior-level fixes (tool strategy, decision policy, sequencing, and fallback logic),
not code changes.
Return strict JSON with this shape:
{
  "summary": "short summary",
  "recommendations": [
    {
      "case_id": "...",
      "primary_failure_mode": "...",
      "target_prompt_file": "system.yaml|tools.yaml|eval.yaml",
      "suggested_prompt_patch": "exact replacement/addition text",
      "rationale": "...",
      "expected_impact": "..."
    }
  ]
}
""".strip()


def _build_case_payload(case: dict[str, Any]) -> dict[str, Any]:
    return {
        "case_id": case.get("case_id"),
        "query": case.get("query"),
        "response": case.get("response"),
        "chat_profile": case.get("chat_profile"),
        "prompt_logical_profile": case.get("prompt_logical_profile"),
        "prompt_manifest": case.get("prompt_manifest"),
        "deterministic_metrics": case.get("deterministic_metrics"),
        "ai_judge_metrics": case.get("ai_judge_metrics"),
        "actual_tool_calls": case.get("actual_tool_calls"),
        "expected_tools": case.get("expected_tools"),
        "required_tools": case.get("required_tools"),
        "forbidden_tools": case.get("forbidden_tools"),
        "tool_names_available": case.get("tool_names_available"),
    }


def _call_recommender(case_payload: dict[str, Any]) -> dict[str, Any]:
    client = AzureOpenAI(
        azure_endpoint=llm_endpoint,
        api_key=llm_api_key,
        api_version=llm_api_version,
    )

    user_payload = {
        "task": "Recommend prompt improvements for this failed scenario.",
        "run_prompt_versions": prompt_versions,
        "current_prompt_files": {
            "system.yaml": prompt_file_text.get("system", ""),
            "tools.yaml": prompt_file_text.get("tools", ""),
            "eval.yaml": prompt_file_text.get("eval", ""),
        },
        "failed_case": case_payload,
    }

    resp = client.chat.completions.create(
        model=llm_deployment,
        temperature=0.2,
        messages=[
            {"role": "system", "content": SYSTEM_INSTRUCTION},
            {"role": "user", "content": json.dumps(user_payload, ensure_ascii=False)},
        ],
        response_format={"type": "json_object"},
    )

    text = (resp.choices[0].message.content or "{}").strip()
    try:
        return json.loads(text)
    except Exception:
        return {
            "summary": "Model did not return valid JSON.",
            "recommendations": [{
                "case_id": case_payload.get("case_id"),
                "primary_failure_mode": "parse_error",
                "target_prompt_file": "system.yaml",
                "suggested_prompt_patch": text,
                "rationale": "Raw model output retained for manual review.",
                "expected_impact": "unknown"
            }],
        }


recommendation_rows: list[dict[str, Any]] = []
raw_llm_outputs: list[dict[str, Any]] = []

if not failure_candidates:
    print("No failed scenarios found in latest run.")
elif not RUN_LLM_RECOMMENDER:
    print("RUN_LLM_RECOMMENDER is False; skipping LLM recommendation generation.")
elif not all([llm_endpoint, llm_api_key, llm_api_version, llm_deployment]):
    print("Missing one or more LLM environment variables; cannot run recommender.")
else:
    selected = failure_candidates[:MAX_FAILURES_TO_ANALYZE]
    print(f"Analyzing {len(selected)} failed scenario(s) with LLM...")

    for case in selected:
        case_payload = _build_case_payload(case)
        output = _call_recommender(case_payload)
        raw_llm_outputs.append({"case_id": case.get("case_id"), "output": output})

        recs = output.get("recommendations") if isinstance(output, dict) else []
        if not isinstance(recs, list):
            recs = []

        for rec in recs:
            if not isinstance(rec, dict):
                continue
            recommendation_rows.append(
                {
                    "case_id": rec.get("case_id") or case.get("case_id"),
                    "primary_failure_mode": rec.get("primary_failure_mode"),
                    "target_prompt_file": rec.get("target_prompt_file"),
                    "suggested_prompt_patch": rec.get("suggested_prompt_patch"),
                    "rationale": rec.get("rationale"),
                    "expected_impact": rec.get("expected_impact"),
                }
            )

recommendations_df = pd.DataFrame(recommendation_rows)

display(recommendations_df)
print(f"Generated {len(recommendations_df)} recommendation row(s).")

Analyzing 5 failed scenario(s) with LLM...


,case_id,primary_failure_mode,target_prompt_file,suggested_prompt_patch,rationale,expected_impact
0,SQL-001,Inappropriate tool call sequence: read_query c...,system.yaml,Add explicit instruction in the Readiness work...,The current system prompt encourages discovery...,"The agent will adopt the proper SQL workflow, ..."
1,SQL-004,Missing required schema_name parameter in desc...,tools.yaml,Add explicit instruction in the describe_table...,The describe_table tool requires both schema_n...,Improved tool call accuracy by ensuring correc...
2,SQL-006,Excessive and unnecessary tool calls causing i...,system.yaml,Add explicit guidance to minimize exploratory ...,The assistant made multiple redundant list_vie...,"Reduced unnecessary tool calls, improved tool ..."
3,SQL-007,Skipped required schema inspection (describe_t...,system.yaml,Add explicit instruction in the Readiness work...,The evaluation rubric and tools.yaml specify a...,"Improved tool call sequencing compliance, high..."
4,SQL-008,Failure to fulfill non-technical user content ...,system.yaml,"Add a communication rule to allow light, appro...",The current system prompt strictly limits resp...,Improved intent resolution and task adherence ...


Generated 5 recommendation row(s).


In [6]:
OUTPUT_PATH = latest_run_dir / "prompt_improvement_recommendations.json"

output_payload = {
    "latest_run_dir": str(latest_run_dir),
    "prompt_versions": prompt_versions,
    "failure_count": len(failure_candidates),
    "analyzed_count": min(len(failure_candidates), MAX_FAILURES_TO_ANALYZE),
    "recommendations": recommendation_rows,
    "raw_llm_outputs": raw_llm_outputs,
}

with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(output_payload, f, ensure_ascii=False, indent=2)

print(f"Saved recommendations to: {OUTPUT_PATH}")

Saved recommendations to: results\traces-20260305-175355__model-gpt-4.1-mini__temp-default__top-p-default\prompt_improvement_recommendations.json


In [7]:
# Consolidate recommendations by target prompt file and build patch drafts

if recommendations_df.empty:
    print("No recommendations available to consolidate.")
else:
    consolidated_rows: list[dict[str, Any]] = []
    patch_drafts: dict[str, str] = {}

    for target_file, group in recommendations_df.groupby("target_prompt_file", dropna=False):
        file_key = str(target_file or "unspecified")
        group = group.reset_index(drop=True)

        patch_sections: list[str] = []
        for i, rec in group.iterrows():
            case_id = rec.get("case_id", "unknown")
            failure_mode = rec.get("primary_failure_mode", "unspecified")
            rationale = rec.get("rationale", "")
            impact = rec.get("expected_impact", "")
            patch_text = rec.get("suggested_prompt_patch", "")

            section = (
                f"## Recommendation {i + 1} (case {case_id})\n"
                f"Failure mode: {failure_mode}\n"
                f"Rationale: {rationale}\n"
                f"Expected impact: {impact}\n\n"
                f"Suggested prompt patch:\n{patch_text}\n"
            )
            patch_sections.append(section)

        consolidated_text = "\n\n".join(patch_sections)
        patch_drafts[file_key] = consolidated_text

        consolidated_rows.append(
            {
                "target_prompt_file": file_key,
                "recommendation_count": len(group),
                "case_ids": ", ".join([str(x) for x in group["case_id"].tolist()]),
                "failure_modes": ", ".join(sorted(set([str(x) for x in group["primary_failure_mode"].tolist()]))),
                "consolidated_patch_preview": consolidated_text[:500] + ("..." if len(consolidated_text) > 500 else ""),
            }
        )

    consolidated_df = pd.DataFrame(consolidated_rows).sort_values("recommendation_count", ascending=False)
    display(consolidated_df)

    print("\n=== Consolidated Prompt Patch Drafts ===")
    for file_key, draft in patch_drafts.items():
        print(f"\n--- {file_key} ---\n")
        print(draft)

    consolidated_output_path = latest_run_dir / "prompt_improvement_consolidated_patches.json"
    with consolidated_output_path.open("w", encoding="utf-8") as f:
        json.dump(
            {
                "latest_run_dir": str(latest_run_dir),
                "generated_from": str(OUTPUT_PATH),
                "by_target_file": patch_drafts,
            },
            f,
            ensure_ascii=False,
            indent=2,
        )

    print(f"\nSaved consolidated patch drafts to: {consolidated_output_path}")

,target_prompt_file,recommendation_count,case_ids,failure_modes,consolidated_patch_preview
0,system.yaml,4,"SQL-001, SQL-006, SQL-007, SQL-008",Excessive and unnecessary tool calls causing i...,## Recommendation 1 (case SQL-001)\nFailure mo...
1,tools.yaml,1,SQL-004,Missing required schema_name parameter in desc...,## Recommendation 1 (case SQL-004)\nFailure mo...



=== Consolidated Prompt Patch Drafts ===

--- system.yaml ---

## Recommendation 1 (case SQL-001)
Failure mode: Inappropriate tool call sequence: read_query calls made before describe_table for schema inspection
Rationale: The current system prompt encourages discovery via list_views but does not explicitly mandate describe_table before read_query. The agent made multiple read_query calls without confirming schema details, violating the recommended discovery->schema inspection->query execution workflow. This can lead to inefficient queries, missed data, or incorrect assumptions about available columns and filters.
Expected impact: The agent will adopt the proper SQL workflow, reducing premature or blind read_query calls. This will improve tool call sequencing, increase coverage and accuracy of queries, and reduce unnecessary or ineffective queries. It will also improve evaluation scores related to tool sequence adherence.

Suggested prompt patch:
Add explicit instruction in the Readin

In [8]:
# Generate full updated prompt files (complete candidates) from consolidated recommendations

GENERATE_FULL_PROMPT_FILES = True
CANDIDATE_DIR = latest_run_dir / "prompt_file_candidates"
CANDIDATE_DIR.mkdir(parents=True, exist_ok=True)

FULL_FILE_SYSTEM_INSTRUCTION = """
You are a meticulous YAML prompt-file editor.
Given the CURRENT file content and consolidated recommendations, produce a full updated file.
Requirements:
- Return valid YAML text for the entire file.
- Preserve existing structure unless recommendations require change.
- Apply recommendations explicitly and concretely.
- Do not include markdown code fences.

CRITICAL FOR system.yaml:
- Keep top-level keys: schema_version, description, profiles.
- Keep schema_version as integer.
- Keep profiles as a mapping.
- Preserve existing logical profile set and order (e.g., sql, search, hybrid).
- Keep each profile as object containing id, version, and text.
- Keep id/version/text as non-empty strings.

CRITICAL FOR tools.yaml:
- Keep top-level keys: schema_version, description, tools.
- Keep schema_version as integer.
- Keep tools as a list of objects.
- Keep each tool object keys: tool_name, version, enabled_profiles, description, usage_rules, examples.
- Preserve existing tool_name set and order; do not add/remove/rename tools.
- Preserve enabled_profiles as list values from {sql, search, hybrid}.
- Keep examples as list of objects with user_prompt and expected.

Return strict JSON:
{
  "full_updated_file": "<entire YAML file text>",
  "notes": "brief summary of what changed"
}
""".strip()


def _build_system_schema_contract(current_system_yaml_text: str) -> dict[str, Any]:
    import yaml

    contract: dict[str, Any] = {
        "required_top_level_keys": ["schema_version", "description", "profiles"],
        "required_profile_keys": ["id", "version", "text"],
        "required_profile_names": [],
        "required_profile_order": [],
    }

    try:
        parsed = yaml.safe_load(current_system_yaml_text) or {}
        if isinstance(parsed, dict):
            profiles = parsed.get("profiles") or {}
            if isinstance(profiles, dict):
                names = list(profiles.keys())
                contract["required_profile_names"] = names
                contract["required_profile_order"] = names
    except Exception:
        pass

    return contract


def _build_tools_schema_contract(current_tools_yaml_text: str) -> dict[str, Any]:
    import yaml

    contract: dict[str, Any] = {
        "required_top_level_keys": ["schema_version", "description", "tools"],
        "required_tool_keys": ["tool_name", "version", "enabled_profiles", "description", "usage_rules", "examples"],
        "allowed_profiles": ["sql", "search", "hybrid"],
        "required_tool_names": [],
        "required_tool_order": [],
    }

    try:
        parsed = yaml.safe_load(current_tools_yaml_text) or {}
        if isinstance(parsed, dict):
            tools = parsed.get("tools") or []
            if isinstance(tools, list):
                names = []
                for item in tools:
                    if isinstance(item, dict) and isinstance(item.get("tool_name"), str):
                        names.append(item.get("tool_name"))
                contract["required_tool_names"] = names
                contract["required_tool_order"] = names
    except Exception:
        pass

    return contract


def _validate_system_candidate(candidate_text: str, current_system_yaml_text: str) -> tuple[bool, str]:
    import yaml

    required_top = {"schema_version", "description", "profiles"}
    required_profile_keys = {"id", "version", "text"}

    try:
        current = yaml.safe_load(current_system_yaml_text) or {}
        cand = yaml.safe_load(candidate_text) or {}
    except Exception as exc:
        return False, f"YAML parse failed: {exc}"

    if not isinstance(cand, dict):
        return False, "Candidate system.yaml is not a mapping"

    if not required_top.issubset(set(cand.keys())):
        return False, "Missing one or more required top-level keys"

    if not isinstance(cand.get("schema_version"), int):
        return False, "schema_version must be an integer"

    if not isinstance(cand.get("description"), str) or not cand.get("description", "").strip():
        return False, "description must be a non-empty string"

    profiles = cand.get("profiles")
    if not isinstance(profiles, dict) or not profiles:
        return False, "profiles must be a non-empty mapping"

    current_profiles = current.get("profiles") if isinstance(current, dict) else None
    current_profile_names: list[str] = []
    if isinstance(current_profiles, dict):
        current_profile_names = list(current_profiles.keys())

    candidate_profile_names = list(profiles.keys())
    if current_profile_names and candidate_profile_names != current_profile_names:
        return False, "profile set/order changed from current system.yaml"

    for profile_name, profile_body in profiles.items():
        if not isinstance(profile_body, dict):
            return False, f"profiles.{profile_name} must be an object"

        missing = required_profile_keys - set(profile_body.keys())
        if missing:
            return False, f"profiles.{profile_name} missing keys: {sorted(missing)}"

        pid = profile_body.get("id")
        pver = profile_body.get("version")
        ptxt = profile_body.get("text")

        if not isinstance(pid, str) or not pid.strip():
            return False, f"profiles.{profile_name}.id must be non-empty string"
        if not isinstance(pver, str) or not pver.strip():
            return False, f"profiles.{profile_name}.version must be non-empty string"
        if not isinstance(ptxt, str) or not ptxt.strip():
            return False, f"profiles.{profile_name}.text must be non-empty string"

    return True, "ok"


def _validate_tools_candidate(candidate_text: str, current_tools_yaml_text: str) -> tuple[bool, str]:
    import yaml

    required_top = {"schema_version", "description", "tools"}
    required_tool_keys = {"tool_name", "version", "enabled_profiles", "description", "usage_rules", "examples"}
    allowed_profiles = {"sql", "search", "hybrid"}

    try:
        current = yaml.safe_load(current_tools_yaml_text) or {}
        cand = yaml.safe_load(candidate_text) or {}
    except Exception as exc:
        return False, f"YAML parse failed: {exc}"

    if not isinstance(cand, dict):
        return False, "Candidate tools.yaml is not a mapping"

    if not required_top.issubset(set(cand.keys())):
        return False, "Missing one or more required top-level keys"

    if not isinstance(cand.get("schema_version"), int):
        return False, "schema_version must be an integer"

    tools = cand.get("tools")
    if not isinstance(tools, list) or not tools:
        return False, "tools must be a non-empty list"

    current_tools = current.get("tools") if isinstance(current, dict) else None
    current_tool_names = []
    if isinstance(current_tools, list):
        for item in current_tools:
            if isinstance(item, dict) and isinstance(item.get("tool_name"), str):
                current_tool_names.append(item.get("tool_name"))

    candidate_tool_names = []
    for idx, item in enumerate(tools):
        if not isinstance(item, dict):
            return False, f"tools[{idx}] is not an object"

        missing = required_tool_keys - set(item.keys())
        if missing:
            return False, f"tools[{idx}] missing keys: {sorted(missing)}"

        tn = item.get("tool_name")
        if not isinstance(tn, str) or not tn.strip():
            return False, f"tools[{idx}].tool_name invalid"
        candidate_tool_names.append(tn)

        version = item.get("version")
        if not isinstance(version, str) or not version.strip():
            return False, f"tools[{idx}].version invalid"

        enabled_profiles = item.get("enabled_profiles")
        if not isinstance(enabled_profiles, list) or not enabled_profiles:
            return False, f"tools[{idx}].enabled_profiles invalid"
        for profile in enabled_profiles:
            if profile not in allowed_profiles:
                return False, f"tools[{idx}].enabled_profiles contains unsupported profile '{profile}'"

        usage_rules = item.get("usage_rules")
        if not isinstance(usage_rules, list) or not usage_rules:
            return False, f"tools[{idx}].usage_rules invalid"

        examples = item.get("examples")
        if not isinstance(examples, list) or not examples:
            return False, f"tools[{idx}].examples invalid"
        for j, ex in enumerate(examples):
            if not isinstance(ex, dict):
                return False, f"tools[{idx}].examples[{j}] must be object"
            if not isinstance(ex.get("user_prompt"), str) or not ex.get("user_prompt", "").strip():
                return False, f"tools[{idx}].examples[{j}].user_prompt invalid"
            if not isinstance(ex.get("expected"), str) or not ex.get("expected", "").strip():
                return False, f"tools[{idx}].examples[{j}].expected invalid"

    if current_tool_names and candidate_tool_names != current_tool_names:
        return False, "tool_name set/order changed from current tools.yaml"

    return True, "ok"


def _generate_full_file_candidate(target_file: str, current_text: str, consolidated_patch: str) -> dict[str, Any]:
    client = AzureOpenAI(
        azure_endpoint=llm_endpoint,
        api_key=llm_api_key,
        api_version=llm_api_version,
    )

    payload = {
        "target_prompt_file": target_file,
        "run_prompt_versions": prompt_versions,
        "current_file_content": current_text,
        "consolidated_recommendations": consolidated_patch,
    }

    if target_file == "system.yaml":
        payload["system_schema_contract"] = _build_system_schema_contract(current_text)
    if target_file == "tools.yaml":
        payload["tools_schema_contract"] = _build_tools_schema_contract(current_text)

    resp = client.chat.completions.create(
        model=llm_deployment,
        temperature=0.1,
        messages=[
            {"role": "system", "content": FULL_FILE_SYSTEM_INSTRUCTION},
            {"role": "user", "content": json.dumps(payload, ensure_ascii=False)},
        ],
        response_format={"type": "json_object"},
    )

    raw = (resp.choices[0].message.content or "{}").strip()
    try:
        parsed = json.loads(raw)
    except Exception:
        parsed = {
            "full_updated_file": current_text,
            "notes": "Failed to parse model output; returned original file content.",
            "raw_model_output": raw,
        }

    if not isinstance(parsed, dict):
        parsed = {"full_updated_file": current_text, "notes": "Unexpected model output type."}

    if not parsed.get("full_updated_file"):
        parsed["full_updated_file"] = current_text
        parsed["notes"] = (parsed.get("notes") or "") + " Fallback used original file content."

    if target_file == "system.yaml":
        ok, reason = _validate_system_candidate(str(parsed.get("full_updated_file") or ""), current_text)
        if not ok:
            parsed["full_updated_file"] = current_text
            parsed["notes"] = (parsed.get("notes") or "") + f" Candidate rejected by schema validator: {reason}. Fallback used original file content."

    if target_file == "tools.yaml":
        ok, reason = _validate_tools_candidate(str(parsed.get("full_updated_file") or ""), current_text)
        if not ok:
            parsed["full_updated_file"] = current_text
            parsed["notes"] = (parsed.get("notes") or "") + f" Candidate rejected by schema validator: {reason}. Fallback used original file content."

    return parsed


full_file_candidates: list[dict[str, Any]] = []

if not GENERATE_FULL_PROMPT_FILES:
    print("GENERATE_FULL_PROMPT_FILES=False; skipping full-file generation.")
elif not all([llm_endpoint, llm_api_key, llm_api_version, llm_deployment]):
    print("Missing LLM env vars; cannot generate full prompt file candidates.")
elif "patch_drafts" not in globals() or not patch_drafts:
    print("No consolidated patch drafts found. Run consolidation cell first.")
else:
    for target_file, consolidated_patch in patch_drafts.items():
        if target_file not in {"system.yaml", "tools.yaml", "eval.yaml"}:
            continue

        current_text = prompt_file_text.get(target_file.replace(".yaml", ""), "")
        result = _generate_full_file_candidate(target_file, current_text, consolidated_patch)

        candidate_path = CANDIDATE_DIR / target_file
        candidate_path.write_text(str(result.get("full_updated_file") or ""), encoding="utf-8")

        full_file_candidates.append(
            {
                "target_prompt_file": target_file,
                "candidate_path": str(candidate_path),
                "notes": result.get("notes", ""),
                "chars": len(str(result.get("full_updated_file") or "")),
            }
        )

candidates_df = pd.DataFrame(full_file_candidates)
display(candidates_df)

candidate_manifest_path = latest_run_dir / "prompt_file_candidates_manifest.json"
with candidate_manifest_path.open("w", encoding="utf-8") as f:
    json.dump(
        {
            "latest_run_dir": str(latest_run_dir),
            "candidate_dir": str(CANDIDATE_DIR),
            "files": full_file_candidates,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

print(f"Saved candidate manifest to: {candidate_manifest_path}")

,target_prompt_file,candidate_path,notes,chars
0,system.yaml,results\traces-20260305-175355__model-gpt-4.1-...,Added explicit instructions in the sql profile...,2456
1,tools.yaml,results\traces-20260305-175355__model-gpt-4.1-...,Added explicit usage rule in describe_table to...,5187


Saved candidate manifest to: results\traces-20260305-175355__model-gpt-4.1-mini__temp-default__top-p-default\prompt_file_candidates_manifest.json
